In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc numpy
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Establecer el nivel de optimización del transpilador

<details>
<summary><b>Versiones de paquetes</b></summary>

El código de esta página fue desarrollado con los siguientes requisitos.
Recomendamos usar estas versiones o versiones más recientes.

```
qiskit[all]~=2.3.0
qiskit-ibm-runtime~=0.43.1
```
</details>
Los dispositivos cuánticos reales están sujetos a ruido y errores de puertas, por lo que optimizar los circuitos para reducir su profundidad y número de puertas puede mejorar significativamente los resultados obtenidos al ejecutar esos circuitos.
La función [`generate_preset_pass_manager`](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.generate_preset_pass_manager#qiskit.transpiler.generate_preset_pass_manager) tiene un argumento posicional obligatorio, `optimization_level`, que controla cuánto esfuerzo dedica el transpilador a optimizar los circuitos. Este argumento puede ser un entero que tome uno de los valores 0, 1, 2 o 3.
Los niveles de optimización más altos generan circuitos más optimizados a costa de tiempos de compilación más largos.
La siguiente tabla explica las optimizaciones que se realizan con cada configuración.

<table>
  <thead>
    <tr>
      <th>Nivel de optimización</th>
      <th>Descripción</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td>0</td>
      <td>
        Sin optimización: se usa típicamente para la caracterización del hardware
        - Traducción básica
        - Layout/Enrutamiento: `TrivialLayout`, que selecciona los mismos números de qubit físicos que virtuales e inserta SWAPs para que funcione (usando `SabreSwap`)
      </td>
    </tr>
    <tr>
      <td>1</td>
      <td>
        Optimización ligera:
        -   Layout/Enrutamiento: primero se intenta el layout con `TrivialLayout`. Si se requieren SWAPs adicionales, se encuentra un layout con el número mínimo de SWAPs usando `SabreSwap`, y luego usa `VF2LayoutPostLayout` para intentar seleccionar los mejores qubits del grafo.
        -   `InverseCancellation`
        -   Optimización de puertas de 1 qubit
      </td>
    </tr>
    <tr>
      <td>2</td>
      <td>
        Optimización media:
          - Layout/Enrutamiento: nivel de optimización 1 (sin trivial) + heurística optimizada con mayor
        profundidad de búsqueda y pruebas de la función de optimización. Como no se usa `TrivialLayout`, no se intenta usar los mismos números de qubit físico y virtual.
        -   `CommutativeCancellation`
      </td>
    </tr>
    <tr>
      <td>3</td>
      <td>
        Optimización alta:
        - Nivel de optimización 2 + heurística optimizada en layout/enrutamiento con mayor esfuerzo/pruebas
        - Resíntesis de bloques de dos qubits usando la [Descomposición KAK de Cartan](https://arxiv.org/abs/quant-ph/0507171).
        - Pasadas que rompen la unitariedad:
          * `OptimizeSwapBeforeMeasure`: mueve las mediciones para evitar SWAPs
          * `RemoveDiagonalGatesBeforeMeasure`: elimina puertas antes de las mediciones que no afectarían a los resultados
      </td>
    </tr>
  </tbody>
</table>

## El nivel de optimización en acción
Como las puertas de dos qubits son típicamente la fuente de errores más significativa, podemos cuantificar aproximadamente la "eficiencia hardware" de la transpilación contando el número de puertas de dos qubits en el circuito resultante.
Aquí probaremos los diferentes niveles de optimización en un circuito de entrada compuesto por un unitario aleatorio seguido de una puerta SWAP.

In [1]:
from qiskit import QuantumCircuit
from qiskit.circuit.library import UnitaryGate
from qiskit.quantum_info import Operator, random_unitary

UU = random_unitary(4, seed=12345)
rand_U = UnitaryGate(UU)

qc = QuantumCircuit(2)
qc.append(rand_U, range(2))
qc.swap(0, 1)
qc.draw("mpl", style="iqp")

<Image src="../docs/images/guides/set-optimization/extracted-outputs/81173ebc-8359-48a6-b585-0477907b3b93-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/set-optimization/extracted-outputs/81173ebc-8359-48a6-b585-0477907b3b93-0.svg)

Usaremos el backend simulado `FakeSherbrooke` en nuestros ejemplos. Primero, transpilemos usando el nivel de optimización 0.

In [2]:
from qiskit.transpiler import generate_preset_pass_manager
from qiskit_ibm_runtime.fake_provider import FakeSherbrooke

backend = FakeSherbrooke()

pass_manager = generate_preset_pass_manager(
    optimization_level=0, backend=backend, seed_transpiler=12345
)
qc_t1_exact = pass_manager.run(qc)
qc_t1_exact.draw("mpl", idle_wires=False)

<Image src="../docs/images/guides/set-optimization/extracted-outputs/40cdd173-b437-48b1-8928-741e8411342e-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/set-optimization/extracted-outputs/40cdd173-b437-48b1-8928-741e8411342e-0.svg)

El circuito transpilado tiene seis puertas ECR de dos qubits.

Repetir para el nivel de optimización 1:

In [3]:
from qiskit.transpiler import generate_preset_pass_manager
from qiskit_ibm_runtime.fake_provider import FakeSherbrooke

backend = FakeSherbrooke()

pass_manager = generate_preset_pass_manager(
    optimization_level=1, backend=backend, seed_transpiler=12345
)
qc_t1_exact = pass_manager.run(qc)
qc_t1_exact.draw("mpl", idle_wires=False)

<Image src="../docs/images/guides/set-optimization/extracted-outputs/2dab5def-a017-42e9-92d6-e043ac4065b2-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/set-optimization/extracted-outputs/2dab5def-a017-42e9-92d6-e043ac4065b2-0.svg)

El circuito transpilado sigue teniendo seis puertas ECR, pero el número de puertas de un solo qubit se ha reducido.

Repetir para el nivel de optimización 2:

In [4]:
pass_manager = generate_preset_pass_manager(
    optimization_level=2, backend=backend, seed_transpiler=12345
)
qc_t2_exact = pass_manager.run(qc)
qc_t2_exact.draw("mpl", idle_wires=False)

<Image src="../docs/images/guides/set-optimization/extracted-outputs/77d76048-b1e8-4225-b35f-80dc9d458e8d-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/set-optimization/extracted-outputs/77d76048-b1e8-4225-b35f-80dc9d458e8d-0.svg)

Esto produce los mismos resultados que el nivel de optimización 1. Ten en cuenta que aumentar el nivel de optimización no siempre marca la diferencia.

Repetir de nuevo, con el nivel de optimización 3:

In [5]:
pass_manager = generate_preset_pass_manager(
    optimization_level=3, backend=backend, seed_transpiler=12345
)
qc_t3_exact = pass_manager.run(qc)
qc_t3_exact.draw("mpl", idle_wires=False)

<Image src="../docs/images/guides/set-optimization/extracted-outputs/4109d0e2-df37-4850-8409-6b860c48595c-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/set-optimization/extracted-outputs/4109d0e2-df37-4850-8409-6b860c48595c-0.svg)

Ahora solo hay tres puertas ECR. Obtenemos este resultado porque en el nivel de optimización 3, Qiskit intenta re-sintetizar bloques de puertas de dos qubits, y cualquier puerta de dos qubits puede implementarse usando como máximo tres puertas ECR. Podemos obtener incluso menos puertas ECR si establecemos `approximation_degree` a un valor menor que 1, permitiendo al transpilador hacer aproximaciones que pueden introducir algo de error en la descomposición de puertas (ver [Parámetros de uso común para la transpilación](common-parameters#approximation-degree)):

In [6]:
pass_manager = generate_preset_pass_manager(
    optimization_level=3,
    approximation_degree=0.99,
    backend=backend,
    seed_transpiler=12345,
)
qc_t3_approx = pass_manager.run(qc)
qc_t3_approx.draw("mpl", idle_wires=False)

<Image src="../docs/images/guides/set-optimization/extracted-outputs/bf239116-b8bb-42aa-a27a-89206d9e108a-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/set-optimization/extracted-outputs/bf239116-b8bb-42aa-a27a-89206d9e108a-0.svg)

Este circuito tiene solo dos puertas ECR, pero es un circuito aproximado. Para entender en qué se diferencia su efecto del circuito exacto, podemos calcular la fidelidad entre el operador unitario que implementa este circuito y el unitario exacto. Antes de realizar el cálculo, primero reducimos el circuito transpilado, que contiene 127 qubits, a un circuito que solo contiene los qubits activos, que son dos.

In [7]:
import numpy as np


def trace_to_fidelity_2q(trace: float) -> float:
    return (4.0 + trace * trace.conjugate()) / 20.0


# Reduce circuits down to 2 qubits so they are easy to simulate
qc_t3_exact_small = QuantumCircuit.from_instructions(qc_t3_exact)
qc_t3_approx_small = QuantumCircuit.from_instructions(qc_t3_approx)

# Compute the fidelity
exact_fid = trace_to_fidelity_2q(
    np.trace(np.dot(Operator(qc_t3_exact_small).adjoint().data, UU))
)
approx_fid = trace_to_fidelity_2q(
    np.trace(np.dot(Operator(qc_t3_approx_small).adjoint().data, UU))
)
print(
    f"Synthesis fidelity\nExact: {exact_fid:.3f}\nApproximate: {approx_fid:.3f}"
)

Synthesis fidelity
Exact: 1.000+0.000j
Approximate: 0.992+0.000j


Adjusting the optimization level can change other aspects of the circuit too, not just the number of ECR gates. For examples of how setting optimization level changes the layout, see [Representing quantum computers](./represent-quantum-computers).

## Next steps

<Admonition type="tip" title="Recommendations">
    - To learn more about the `generate_preset_passmanager` function, start with the [Transpilation default settings and configuration options](defaults-and-configuration-options) topic.
    - Continue learning about transpilation with the [Transpiler stages](transpiler-stages) topic.
    - Try the [Compare transpiler settings](/docs/guides/circuit-transpilation-settings) guide.
    - Try the [Build repetition codes](/docs/tutorials/repetition-codes) tutorial.
    - See the [Transpile API documentation.](/docs/api/qiskit/transpiler)
</Admonition>